In [2]:
# objective: generate the particle size structure for the CMIP6 models
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd
import glob,os,subprocess
import IPython.display as display
import math
from datetime import datetime as dt
%matplotlib inline

In [ ]:
from scipy.stats import linregress

In [ ]:
carbon_molar_mass = 12.011 # g/mol

## Pre step: read files, create lists for each group of files that we'll need.

In [ ]:
# create pattern to load the files 
#model_list = [ 'cesm2', 'cmcc', 'cnrm', 'gfdl',  'giss', 'ipsl', 'ukesm'] #

depth_list = ['0-100', '100-200']

p_categories = ['phymisc', 'phydiat', 'zmeso', 'zmicro'] # 'phymisc', removing this to see what happens with slope

phyto_categories = ['phydiat', 'phydiaz', 'phypico', 'phymisc', 'phycalc']

zoo_categories = ['zmeso', 'zmicro']

models_same = ['cmcc', 'cnrm', 'ipsl', 'ukesm']

pathBase='/work/jyl/proj/CMIP6_size/CMIP6_output_regular_grid' #path to folder

files = os.listdir(pathBase)
files=['.'.join([os.path.join(pathBase,f)]) for f in files]
files_cmcc = [model for model in files if 'cmcc' in model]

chla_hist = '/work/jyl/proj/CMIP6_size/CMIP6_output_regular_grid/cmcc_hist_schl_gr_monthly_1965_2014.nc'
chla_ssp5 = '/work/jyl/proj/CMIP6_size/CMIP6_output_regular_grid/cmcc_ssp585_schl_gr_monthly_2015_2100.nc'

print (files_cmcc)

In [ ]:
files_ssp585 = [x for x in files_cmcc if 'ssp585' in x and 'chlos' not in x and 'mld' not in x and 'tos' not in x and 'schl' not in x and 'phyc' not in x]

files_hist = [x for x in files_cmcc if 'hist' in x and 'chlos' not in x and 'mld' not in x and 'tos' not in x and 'schl' not in x and 'phyc' not in x]
files_hist


In [ ]:
chl_hist=xr.open_mfdataset(chla_hist)
chl_hist=chl_hist.sel(time= slice(1985.0, 2015))
chl_ssp5 =xr.open_mfdataset(chla_ssp5)
chl_ssp5=chl_ssp5.sel(time= slice(2070.0, 2100))
chl_hist#.mean(dim=('lat', 'lon')).schl.plot()

## Change time format from 365_day to day-month-year

In [ ]:
def change_date_format(chl_hist):
    chl_hist=chl_hist.assign_coords(year = np.trunc(chl_hist.time).astype(int))#.astype(int)
    chl_hist=chl_hist.assign_coords(day = ((chl_hist.time - chl_hist.year)*365).astype(int))
    chl_hist = chl_hist.where(chl_hist.day !=0, drop=True)
    chl_hist=chl_hist.assign_coords(year = chl_hist.year.astype(str))
    chl_hist=chl_hist.assign_coords(day = np.char.zfill(chl_hist.day.astype(str),3))
    chl_hist=chl_hist.assign_coords(time = np.char.add(np.char.add(chl_hist.year, '-'),chl_hist.day))
    chl_hist= chl_hist.drop_dims('day')
    chl_hist = chl_hist.drop('year', dim=None)
    chl_hist=chl_hist.assign_coords(time = [dt.strptime(x, '%Y-%j') for x in chl_hist.time.values])
    chl_hist = chl_hist.sortby('time', ascending = True)
    return (chl_hist)

In [ ]:
chl_hist = change_date_format(chl_hist)
chl_ssp5 = change_date_format(chl_ssp5)
chl_hist.to_netcdf('/work/m1c/CMIP6_biome_PSS_data/cmcc_hist_schl_gr_monthly_1985_2014.nc')
chl_ssp5.to_netcdf('/work/m1c/CMIP6_biome_PSS_data/cmcc_ssp585_schl_gr_monthly_2070_2100.nc')
chl_hist

## Pre step: combine the different files of the plankton into one netcdf file

In [ ]:
# combine historical data in one file and add variables by depth, and remove old datasets
ds_hist= xr.open_mfdataset(files_hist, combine = 'by_coords')
ds_hist = ds_hist.drop_vars('zmeso200')
for i in [ 'phydiat', 'phymisc', 'zmicro', 'zmeso']:
    ds_hist[i +'_0_200'] = ds_hist[i +'_100'] + ds_hist[i+'_200']
    ds_hist = ds_hist.drop_vars([i+'_100', i+'_200'])
ds_hist=ds_hist.sel(time= slice(1985.0, 2015)) # some data earlier than 1984 had to be removed
ds_hist=ds_hist.sortby('lat', ascending=True)

In [ ]:
#necessary step to make sure that cells with Nans are the same across the PFTs
ds_hist_mask = ~(np.isnan(ds_hist.phydiat_0_200) | np.isnan(ds_hist.phymisc_0_200) | np.isnan(ds_hist.zmicro_0_200) | np.isnan(ds_hist.zmeso_0_200))
ds_hist = ds_hist.where(ds_hist_mask)

In [ ]:
ds_hist.mean(dim=('time')).zmeso_0_200.plot()

## Step 1b. Compute total global carbon

In [ ]:

area_path = '/work/m1c/GFDL_files/ocean_cobalt_omip_tracers_month_z_1x1deg.static.nc'
area_grid = xr.open_dataset(area_path)
# coordinates need to match with the previous files
area_grid.coords['lon'] = (area_grid.coords['lon'] + 180) % 360 - 180
area_grid = area_grid.sortby(area_grid.lon)
#area_grid = area_grid.sortby(area_grid.lat) 
#area_grid = area_grid.assign_coords(lat=(area_grid.lat * -1))

area_grid

In [ ]:
# dictionary with total carbon
varlist=['phydiat', 'phymisc', 'zmicro', 'zmeso']
total_carbon_gC={}
total_carbon_gC['globalSum_gC']={}
    
for i in varlist:
    total_carbon_gC['globalSum_gC'][i] = (ds_hist[i + '_0_200'].mean(dim='time') * area_grid.areacello).sum(dim=['lon','lat']).values # conversion to g C already happened
total_carbon_gC

In [ ]:
# transform the dictionary to a dataframe
biomass=pd.DataFrame.from_dict(total_carbon_gC)
biomass=biomass.reset_index()
biomass.columns.values[0]='name'
biomass

## Step3: complete table with size classes 

In [ ]:
sizes=np.linspace(np.log10(2),np.log10(35.*1000.),num=1000,endpoint=False)
#10**(sizes)
#sizes

In [ ]:
dd=np.diff(sizes)[0]
sizes_min=10**(sizes-dd/2)
sizes_max=10**(sizes+dd/2)
size_range=[str(np.round(10**s,3))+': '+'['+str(np.round(x,3))+','+str(np.round(y,3))+')' for s,x,y in zip(sizes,sizes_min,sizes_max)]
#size_range

In [ ]:
sizedf=pd.DataFrame([10**sizes]).transpose()
sizedf.columns=['sizes']
sizedf['phyto']= None
sizedf['zoo']= None
#sizedf.loc[sizedf.sizes < 10,'phyto']='smp'
sizedf.loc[(sizedf.sizes > 2) & (sizedf.sizes < 20),'phyto']='phymisc'
sizedf.loc[(sizedf.sizes > 20) & (sizedf.sizes < 200),'phyto']='phydiat' ##Playing with diatom size 
sizedf.loc[(sizedf.sizes > 20) & (sizedf.sizes < 200),'zoo']='zmicro'
sizedf.loc[(sizedf.sizes > 200) & (sizedf.sizes < 35000),'zoo']='zmeso'
#sizedf.loc[(sizedf.sizes > 2000) & (sizedf.sizes < 20000),'zoo']='lgz'
##this step is necessary to remove the empty categories when we are not considering phymisc



sizedf = sizedf.reset_index(drop=True)

#sizedf

In [ ]:
# determine the degree of overlap between size bins. In COBALT (and perhaps other models) the degree of overlap might need to be determined following 
# Jessica's method

# the goal is to have repeated entries for a size class if it occurs across plantkon types:
sdfm = pd.melt(sizedf, id_vars='sizes',var_name='type',value_name='name')
sdfm = sdfm.dropna().reset_index(drop=True)
pd.set_option('display.max_rows',80)
#sdfm

In [ ]:
# assign biovolume and amount of acrbon to each size bin
import math

sdfm['biovolume_um3']=(4/3)*math.pi*(sdfm.sizes/2)**3

sdfm['mg_carbon']=0
#non-diatoms, Menden-Deuer and Lessard 2000, double check these with Jessica, to ask if might be appropiate to use dinoflagellate fits
#also, why 0.216?? notice phymisc as non diatoms

# phymisc, which are non diatoms, treated as protists, Menden-Deuer and Lessard 2000
# < 3000 um3 biovolume
tmp=sdfm.loc[(sdfm.name=='phymisc') & (sdfm.biovolume_um3 < 3000),'biovolume_um3']
sdfm.loc[(sdfm.name=='phymisc') & (sdfm.biovolume_um3 < 3000),'mg_carbon']=10**(-0.583 + 0.860 * np.log10(tmp)) * 1e-9
# > 3000 um3 biovolume OJO under current size classes, all phydiat will fit here. This might be a big source of biomass overestimation
tmp=sdfm.loc[(sdfm.name=='phymisc') & (sdfm.biovolume_um3 >= 3000),'biovolume_um3']
sdfm.loc[(sdfm.name=='phymisc') & (sdfm.biovolume_um3 >= 3000),'mg_carbon']=10**(-0.665 + 0.939 * np.log10(tmp)) * 1e-9


# zmicro, treated as protists, Menden-Deuer and Lessard 2000
# < 3000 um3 biovolume
tmp=sdfm.loc[(sdfm.name=='zmicro') & (sdfm.biovolume_um3 < 3000),'biovolume_um3']
sdfm.loc[(sdfm.name=='zmicro') & (sdfm.biovolume_um3 < 3000),'mg_carbon']=10**(-0.583 + 0.860 * np.log10(tmp)) * 1e-9
# > 3000 um3 biovolume OJO under current size classes, all phydiat will fit here. This might be a big source of biomass overestimation
tmp=sdfm.loc[(sdfm.name=='zmicro') & (sdfm.biovolume_um3 >= 3000),'biovolume_um3']
sdfm.loc[(sdfm.name=='zmicro') & (sdfm.biovolume_um3 >= 3000),'mg_carbon']=10**(-0.665 + 0.939 * np.log10(tmp)) * 1e-9


#sdfm.loc[sdfm.name=='phymisc','mg_carbon']=0.216 * sdfm.loc[sdfm.name=='phymisc','biovolume_um3']**0.939 * 1e-9
#sdfm.loc[sdfm.name=='zmicro','mg_carbon']=0.216 * sdfm.loc[sdfm.name=='zmicro','biovolume_um3']**0.939 * 1e-9

# diatoms, Menden-Deuer and Lessard 2000
# < 3000 um3 biovolume
tmp=sdfm.loc[(sdfm.name=='phydiat') & (sdfm.biovolume_um3 <= 3000),'biovolume_um3']
sdfm.loc[(sdfm.name=='phydiat') & (sdfm.biovolume_um3 <= 3000),'mg_carbon']=10**(-0.541 + 0.811 * np.log10(tmp)) * 1e-9
# > 3000 um3 biovolume OJO under current size classes, all phydiat will fit here. This might be a big source of biomass overestimation
tmp=sdfm.loc[(sdfm.name=='phydiat') & (sdfm.biovolume_um3 > 3000),'biovolume_um3']
sdfm.loc[(sdfm.name=='phydiat') & (sdfm.biovolume_um3 > 3000),'mg_carbon']=10**(-0.933 + 0.881 * np.log10(tmp)) * 1e-9

# mesozooplankton, Pitt et al. 2013 will be deprecated, we will use Kiorboe (2013)/Maas et al. (2021) combo
#sdfm.loc[sdfm.name=='zmeso','mg_carbon']= 0.06281 * (sdfm.loc[sdfm.name=='zmeso','sizes']/1e3)**3
sdfm.loc[sdfm.name=='zmeso','mg_carbon'] = 0.055 * (sdfm.loc[sdfm.name=='zmeso','biovolume_um3']/1e9) # Maas et al. takes milimeters cubed (notice conversion) and returns dry mass in mg
sdfm.loc[sdfm.name=='zmeso','mg_carbon'] = 10**((np.log10(sdfm.loc[sdfm.name=='zmeso','mg_carbon'])-(-0.67))/0.96) # Kiorboe et al. takes dry mass in mg to wet mass in mg
sdfm.loc[sdfm.name=='zmeso','mg_carbon'] = (10**((0.95*np.log10(sdfm.loc[sdfm.name=='zmeso','mg_carbon']))-0.93))# Kiorboe et al. takes wet mass  and returns mass of carbon


len(sdfm)


In [ ]:
sdfm.loc[sdfm.name=='phymisc'].biovolume_um3.max()

## define the linear regressions for the PFTs that can use more than one allomentric relations due to its size range. For CMCC these are phymisc:

## first, phymisc

In [ ]:
df_phymisc = sdfm.loc[sdfm.name=='phymisc'].reset_index()
df_phymisc['pg_carbon'] = df_phymisc['mg_carbon']*1e9
df_phymisc

In [ ]:
slope_phymisc, intercept_phymisc, r_value_phymisc, p_value_phymisc, std_err_phymisc = linregress(x=np.log10(df_phymisc.biovolume_um3),y=np.log10(df_phymisc.pg_carbon))

In [ ]:
slope_phymisc

In [ ]:
intercept_phymisc

## now zmicro

In [ ]:
df_zmicro = sdfm.loc[sdfm.name=='zmicro'].reset_index()
df_zmicro['pg_carbon'] = df_zmicro['mg_carbon']*1e9
df_zmicro

In [ ]:
slope_zmicro, intercept_zmicro, r_value_zmicro, p_value_zmicro, std_err_zmicro = linregress(x=np.log10(df_zmicro.biovolume_um3),y=np.log10(df_zmicro.pg_carbon))

In [ ]:
slope_zmicro

In [ ]:
intercept_zmicro

## Now use these linear regressions to inform the functions from carbon to biovolume:

In [ ]:
def g_carbon_to_biovol(x, v): # allometric relations based on Menden-Deuder and Lessard() for phymisc, zmicro and phydiat,  and Pitt et al. for zmeso. takes grams, converts to picograms
    if v =='phymisc':
        x_biovol = 10**((np.log10(x*1e12)-(intercept_phymisc))/slope_phymisc)
        
    elif v == 'zmicro':
        x_biovol = 10**((np.log10(x*1e12)-(intercept_zmicro))/slope_zmicro)
        
    elif v == 'phydiat':
        x_biovol = 10**((np.log10(x*1e12)-(-0.933))/0.881) # onnly one regressionn for diatoms since all diatoms here are >3000 um3


    elif v == 'zmeso':
        #x_micrometers = (10**((0.33*np.log10(x))-0.6))*10000 # to  ESD in micrometers. Notice that allometric relations for Pitt et al have on the y variable cm and the x variable carbon in grams
        #x_micrometers = 10**((np.log10(x*1e6)-(-0.698))/2.476)# This relationship is for Acartia tonsa from Mauchline (1999) grams of carbon converted to micrograms
        #x_micrometers = 10**((np.log10(x*1e6)-(-5.58))/2.23) # relationship from Rodriguez & Mullin (1986), Takes micrograms and returrs micrometers, does not work, over estimation of meso biomass
        #x_biovol=(4/3)*math.pi*(x_micrometers/2)**3 #from mcirometers to micrometers cubed
        
        #the next approach uses three equations: two from Kiorboe et al. (2013), to go from carbom mass to wet mass and finally to dry mass, and then from Maas et al. 2021 from dry mass to biovolume
        wet_mass = 10**((np.log10(x*1000)-(-0.93))/0.95) # takes miligrams of carbon and returns miligrams of wet weight Kiorboe et al. (2013)
        dry_mass = (10**((0.96*np.log10(wet_mass))-0.67))# takes miligrams of wet mass and returns miligrams of dry mass Kiorboe et al. (2013)
        x_biovol = ((dry_mass)/0.055)*1e9 #Maas et al (2021) appears to set the intercept to 0 and does not use a logarithmic function to relate these variables. Original equation: dry mass = 0.055*biovolume +0. This also includes conversion from mm3 to um3
        #x_biovol = 10**((np.log10(x*1e12)-(-0.665))/0.939)
    return x_biovol

In [ ]:
global_um3_list = []
for n, v in enumerate(biomass.name.unique()):
    global_um3_list.append(g_carbon_to_biovol(biomass.globalSum_gC[n], v))
biomass['globalSum_um3'] = global_um3_list

In [ ]:
biomass

## Step 3. pull together and merge the overlapping size bins. The total global biovolume is split by the size bins


In [ ]:

sdfm['globalSum_um3_split']=0
for s in sdfm.name.unique():
    n=len(sdfm.loc[sdfm.name==s].index)
    print(s, n)
    sdfm.loc[sdfm.name==s,'globalSum_um3_split'] = np.tile(biomass.loc[biomass.name==s,'globalSum_um3']/n,n)
sdfm

In [ ]:
sdfm = sdfm.sort_values(by='biovolume_um3', ascending=True)

In [ ]:
small_increment_biovol = (sdfm['biovolume_um3'][1]-sdfm['biovolume_um3'][0])/2 # small increment is used to define the maximum and minimum of the size range

In [ ]:
# create log-spaced bins for mg_carbon
bins_biovol = np.logspace(np.log10(sdfm['biovolume_um3'].min()-small_increment_biovol), np.log10(sdfm['biovolume_um3'].max()+small_increment_biovol), 51)


# use pandas.cut to bin the data into log-spaced bins
sdfm['biovolume_um3_bin'] = pd.cut(sdfm['biovolume_um3'], bins=bins_biovol, include_lowest=False)
sdfm['bin_centers_biovol'] = sdfm['biovolume_um3_bin'].apply(lambda x:x.mid).astype(float) # this gets the mid point 
sdfm['bin_range_biovol'] = sdfm['biovolume_um3_bin'].apply(lambda x:x.length).astype(float)

#len(sdfm)

In [ ]:
sdfm.head()

In [ ]:
len(sdfm)

In [ ]:
df_grouped_biovol = sdfm.groupby([ 'name', 'biovolume_um3_bin','bin_centers_biovol', 'bin_range_biovol']).agg(biovolume_um3=('biovolume_um3','mean'), globalSum_um3=('globalSum_um3_split','sum'),
                                                                                                             sizes=('sizes','mean'))


df_grouped_biovol = df_grouped_biovol.dropna().sort_values(by = ['sizes']).reset_index()
df_grouped_biovol['NB'] = df_grouped_biovol.globalSum_um3/df_grouped_biovol.bin_range_biovol
df_grouped_biovol.sizes = df_grouped_biovol.sizes.round(1)
df_grouped_biovol



In [ ]:
bin_info = pd.DataFrame({
    'biovolume_um3_bin': np.sort(sdfm['biovolume_um3_bin'].unique()),
    'bin_centers_biovol': np.sort(sdfm['bin_centers_biovol'].unique())})

In [ ]:
bin_info

## Calculate  total biomass and normalized biomass per size class

In [ ]:
lat = ds_hist.lat
lon = ds_hist.lon
time = ds_hist.time
biovol_um3 = bin_info['bin_centers_biovol']
data = np.zeros((len(biovol_um3), len(time), len(lat), len(lon)))
biovolume_all = xr.DataArray(data, coords={'biovol_um3':biovol_um3, 'time':time, 'lat':lat, 'lon':lon},
            dims = ['biovol_um3', 'time', 'lat', 'lon'])


lat_NB = ds_hist.lat
lon_NB = ds_hist.lon
time_NB = ds_hist.time
biovol_um3_NB = bin_info['bin_centers_biovol']
data_NB = np.zeros((len(biovol_um3_NB), len(time_NB), len(lat_NB), len(lon_NB)))
biovolume_all_NB = xr.DataArray(data_NB, coords={'biovol_um3':biovol_um3_NB, 'time':time_NB, 'lat':lat_NB, 'lon':lon_NB},
            dims = ['biovol_um3', 'time', 'lat', 'lon'])

#time = ds.time
#data = np.zeros((len(time), len(mmolC), len(z_t_150m), len(nlat), len(nlon)))
#biomass_all = xr.DataArray(data, coords={'time':time, 'mass_mmolC':mmolC, 'z_t_150m':z_t_150m, 'nlat':nlat, 'nlon':nlon},
#            dims = ['time', 'mass_mmolC', 'z_t_150m', 'nlat', 'nlon'])




biovolume_all.shape

In [ ]:
biovolume_all_NB.shape

In [ ]:
def size_spectra(data_array, dataset, NB=False):
    for v in df_grouped_biovol['name'].unique():
        print(v)
        cobaltvar = v +'_0_200' # REMEMBER: it has to be with the integrated data variables, to remove depth dimensions
        x = dataset[cobaltvar]#.mean(dim='time') # average by time
        x = x.values * carbon_molar_mass # to g C
        x = g_carbon_to_biovol(x, v)

    
        # split up 
        n_split = len(df_grouped_biovol.loc[df_grouped_biovol.name==v].index)

        um_biovolume = df_grouped_biovol.loc[df_grouped_biovol.name==v,'bin_range_biovol'].values
        x_split = x / n_split

    
        x_rep = np.repeat(x_split[np.newaxis,...], n_split, axis=0)
    
        if NB==True:
            x_rep = x_rep / (um_biovolume[:,None,None,None]) # normalized biomass = integrated biomass / size-class

        # units are : um3 / m^2 / um3 ind-1

        print(np.nanmax(x_rep))
        # extract the indices corresponding to the bins where the plankton groups fall
        pft_bins=pd.unique(df_grouped_biovol.where(df_grouped_biovol.name==v).dropna().bin_centers_biovol)
        subset = bin_info[bin_info['bin_centers_biovol'].isin(pft_bins)]
        index_dims = subset.index.values
    
        # check lengths are the same
        if (n_split != len(index_dims)):
            print("Error: Dimension lengths are not the same")
            break
        #normalized biovolume
        biovolume_vals = data_array[index_dims,:,:,:].values
        biovolume_vals = biovolume_vals + x_rep
        # put into matrix
        data_array.values[index_dims,:,:,:] = biovolume_vals
    return(data_array)

In [ ]:
biovolume_all = size_spectra(biovolume_all, ds_hist, NB=False)

In [ ]:
biovolume_all.mean(dim=('lat', 'lon', 'time')).plot()
plt.xscale('log')
plt.yscale('log')

## Get the total biovolume for only the size range included in PSSdb UVP+Zooscan

In [ ]:
biovol_mask = ~np.isnan(biovolume_all.mean(dim=('biovol_um3')))
biovolume_all_subset= biovolume_all.where((biovolume_all['biovol_um3']>826400.0) & (biovolume_all['biovol_um3']<49100000000000.0)).sum(dim=['biovol_um3'])
biovolume_all= biovolume_all.sum(dim=['biovol_um3'])
biovolume_all = biovolume_all.where(biovol_mask)
biovolume_all_subset = biovolume_all_subset.where(biovol_mask)
biovolume_all.mean(dim=('time')).plot()

                

In [ ]:
biovolume_all_subset.mean(dim=('time')).plot()

In [ ]:
biovolume_all_NB = size_spectra(biovolume_all_NB, ds_hist, NB=True)

In [ ]:
biovolume_all_NB.mean(dim=('lat', 'lon', 'time')).plot()
plt.xscale('log')
plt.yscale('log')

In [ ]:
biovolume_all_NB.mean(dim=('time', 'biovol_um3')).plot()

In [ ]:
# removing data from mediterranean and black sea
#biovolume_all.loc[dict(lat=biovolume_all.coords['lat'][(biovolume_all.coords['lat'] >= 30.5) & (biovolume_all.coords['lat'] <= 47.5)],
                                        #lon=biovolume_all.coords['lon'][(biovolume_all.coords['lon'] >= -5.5) & (biovolume_all.coords['lon'] <= 55.5)])]=float('nan') #

In [ ]:
def calculate_size_spectra_slopes(data):
    data.values=np.log10(data.values)
    data = data.assign_coords(biovol_um3=np.log10(data.biovol_um3))
    cov_x = (data.biovol_um3.values - data.biovol_um3.mean().values)[:,None,None,None]
    cov_y = (data.values - data.mean(dim='biovol_um3').values[None,:,:,:])
    cov_xy = cov_x * cov_y
    covariance = np.nansum(cov_xy,axis=0)

    #variance = (data.mass_mgC.values.var() * (len(data.mass_mgC.values) - 1.))
    variance = np.nansum(cov_x**2, axis=0)

    betas = covariance / variance
    intercept_log = data.mean(dim='biovol_um3').values - (betas * data.biovol_um3.mean().values)
    intercept = 10**(data.mean(dim='biovol_um3').values - (betas * data.biovol_um3.mean().values))
    #print(betas.shape)
    #print(intercept.shape)
    #print(data.mass_mgC.values[:, None,None].shape)
    Y_pred = betas*(data.biovol_um3.values[:, None,None,None])+intercept_log
    ss_res = np.nansum((data.values - Y_pred)**2, axis = 0)
    ss_tot = np.nansum((cov_y)**2, axis = 0)
    R2 = 1-(ss_res/ss_tot)
    RMSE = (ss_res/data.biovol_um3.values.shape[0])**0.5
    #print(Y_pred.shape)
    #ss_tot = (data.values - data.mean(dim='mass_mgC').values[None,:,:])**2
    #ss_res = (data.values-Y_pred)
    #R2 = 1 - (ss_res / ss_tot)
    
    
    
    #betas = -betas # convention
    betas[betas == 0] = np.nan
    intercept[intercept ==0] = np.nan
    R2[R2 ==0] = np.nan
    RMSE[RMSE ==0] = np.nan
    
    return betas, intercept, R2, RMSE

In [ ]:
betas_hist, intercept_hist, R2_hist, RMSE_hist = calculate_size_spectra_slopes(biovolume_all_NB)
betas_hist.shape

In [ ]:
np.nanmean(betas_hist)

In [ ]:
np.nanmax(betas_hist)

In [ ]:
np.nanmin(betas_hist)

In [ ]:
biovolume_all_NB.values=10**(biovolume_all_NB.values)

## add total  grid biovolume to the xarray and untransform NB values. The data needs to be coverted to biovolume

In [ ]:
#phymisc=g_carbon_to_biovol(((ds_hist.phymisc_0_200.values)*carbon_molar_mass), 'phymisc')
#phydiat=g_carbon_to_biovol(((ds_hist.phydiat_0_200.values)*carbon_molar_mass), 'phydiat')
#zmicro=g_carbon_to_biovol(((ds_hist.zmicro_0_200.values)*carbon_molar_mass), 'zmicro')
#zmeso=g_carbon_to_biovol(((ds_hist.zmeso_0_200.values)*carbon_molar_mass), 'zmeso')



## Get the slopes for each biome

In [ ]:
# load the biome mask
biome_mask = '/work/jyl/proj/CMIP6_models/ESM_Biomes/CMCC_historical_biomes_x1.nc'
biomes_hist = xr.open_dataset(biome_mask)
biomes_mask = ~np.isnan(biomes_hist.biomes)
biomes_hist['biomes'] = biomes_hist['biomes'].where((biomes_hist['lat'] < 44.5) & (biomes_hist['lat'] > -44.5), 2)
biomes_hist = biomes_hist.where(biomes_mask)
#biomes_hist = biomes_hist.assign_coords(lat=(biomes_hist.lat * -1))
biomes_hist.biomes.plot()


In [ ]:
biomes_hist

In [ ]:

# add slopes to the biomes dataset
biomes_hist['chl'] = (('time', 'lat', 'lon'), chl_hist.schl.values)
biomes_hist['chl'].attrs = {"units": 'kg m-3', 'Description': 'Surface Mass Concentration of Total Phytoplankton expressed as Chlorophyll in Sea Water'}

biomes_hist['NB'] = biovolume_all_NB
biomes_hist['NB'].attrs = {"units": 'um^3 m^-2 m^-3', 'Description': 'normalized biovolume for each size class'}

#biomes_hist['total_biovolume_hist_full'] = biovolume_all
biomes_hist['total_biovolume_hist'] = biovolume_all_subset#(('time', 'lat', 'lon'),biovolume_all.values)
biomes_hist['total_biovolume_hist'].attrs = {"units": 'um^3 ', 'Description': 'total biovolume from the size range of 116-2000 micrometers'}

biomes_hist['betas_hist'] = (('time', 'lat', 'lon'), betas_hist)
biomes_hist['betas_hist'].attrs = {"units": 'm^-2 m^-3', 'Description': 'Normalized biovolume size spectra slope'}

biomes_hist['intercept_hist'] = (('time', 'lat', 'lon'), intercept_hist)
biomes_hist['intercept_hist'].attrs = {"units": 'um^3 m^-2 m^-3', 'Description': 'Normalized biovolume size spectra intercept'}

biomes_hist['R2_hist'] = (('time', 'lat', 'lon'), R2_hist)
biomes_hist['R2_hist'].attrs = { 'Description': 'Normalized biovolume coefficient of determination'}

biomes_hist['RMSE_hist'] = (('time', 'lat', 'lon'), RMSE_hist)
biomes_hist['RMSE_hist'].attrs = {"units": 'um^3 m^-2 m^-3', 'Description': 'Normalized biovolume size spectra Root mean square error'}

#original_biomass_variables
biomes_hist['phymisc'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_hist.phymisc_0_200.values*carbon_molar_mass, 'phymisc'))
biomes_hist['phymisc'].attrs = {"units": 'um^3', 'Description': 'miscellaneous phytoplankton biovolume'}

biomes_hist['phydiat'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_hist.phydiat_0_200.values*carbon_molar_mass, 'phydiat'))
biomes_hist['phydiat'].attrs = {"units": 'um^3', 'Description': 'diatom biovolume'}

biomes_hist['zmicro'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_hist.zmicro_0_200.values*carbon_molar_mass, 'zmicro'))
biomes_hist['zmicro'].attrs = {"units": 'um^3', 'Description': 'microzooplankton biovolume'}

biomes_hist['zmeso'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_hist.zmeso_0_200.values*carbon_molar_mass, 'zmeso'))
biomes_hist['zmeso'].attrs = {"units": 'um^3', 'Description': 'mesozooplankton biovolume'}

biomes_hist['total_biovolume_full']=(('time', 'lat', 'lon'), biomes_hist.phymisc.values+biomes_hist.phydiat.values+biomes_hist.zmicro.values+biomes_hist.zmeso.values)
biomes_hist['total_biovolume_full'].attrs = {"units": 'um^3', 'Description': 'all plankton biovolume'}
                                          
#percentage total
biomes_hist['phymisc_per'] = (('time', 'lat', 'lon'), (biomes_hist.phymisc.values/biomes_hist.total_biovolume_full.values)*100)
biomes_hist['phymisc_per'].attrs = {'Description': 'miscellaneous phytoplankton percentage of total biovolume'}                                        
                                          
biomes_hist['phydiat_per'] = (('time', 'lat', 'lon'),(biomes_hist.phydiat.values/biomes_hist.total_biovolume_full.values)*100)
biomes_hist['phydiat_per'].attrs = {'Description': 'diatom percentage of total biovolume'}                                           
                                          
biomes_hist['zmicro_per'] = (('time', 'lat', 'lon'),(biomes_hist.zmicro.values/biomes_hist.total_biovolume_full.values)*100)
biomes_hist['zmicro_per'].attrs = {'Description': 'microzooplankton percentage of total biovolume'} 

biomes_hist['zmeso_per'] = (('time', 'lat', 'lon'),(biomes_hist.zmeso.values/biomes_hist.total_biovolume_full.values)*100)
biomes_hist['zmeso_per'].attrs = {'Description': 'mesozooplankton percentage of total biovolume'} 

#total biovolume
biomes_hist['phyc'] = (('time', 'lat', 'lon'), biomes_hist.phymisc.values+biomes_hist.phydiat.values)
biomes_hist['phyc'].attrs = {"units": 'um^3', 'Description': 'phytoplankton biovolume'}

biomes_hist['zooc'] = (('time', 'lat', 'lon'), biomes_hist.zmicro.values+biomes_hist.zmeso.values)
biomes_hist['zooc'].attrs = {"units": 'um^3', 'Description': 'zooplankton biovolume'}

#biomes_hist['total_biovolume_hist']= (('time', 'lat', 'lon'), phypico+phydiat+zooc)
biomes_hist

In [ ]:
biomes_hist.mean(dim=('time')).total_biovolume_hist.plot()

## Change time format from 365_day to day-month-year

In [ ]:
biomes_hist=biomes_hist.assign_coords(year = np.trunc(biomes_hist.time).astype(int))#.astype(int)
biomes_hist=biomes_hist.assign_coords(day = ((biomes_hist.time - biomes_hist.year)*365).astype(int))
biomes_hist = biomes_hist.where(biomes_hist.day !=0, drop=True)

biomes_hist=biomes_hist.assign_coords(year = biomes_hist.year.astype(str))
biomes_hist=biomes_hist.assign_coords(day = np.char.zfill(biomes_hist.day.astype(str),3))



In [ ]:
biomes_hist=biomes_hist.assign_coords(time = np.char.add(np.char.add(biomes_hist.year, '-'),biomes_hist.day))
#ds_hist=ds_hist.assign_coords(time = ds_hist.str.cat(ds_hist.year,ds_hist.day, sep = '-'))

In [ ]:
biomes_hist= biomes_hist.drop_dims('day')
biomes_hist = biomes_hist.drop('year', dim=None)

In [ ]:
#sorted(ds_hist.time.values)

In [ ]:
from datetime import datetime as dt
biomes_hist=biomes_hist.assign_coords(time = [dt.strptime(x, '%Y-%j') for x in biomes_hist.time.values])

#ds_hist['time'].dt.strftime('%y%j')

In [ ]:
biomes_hist = biomes_hist.sortby('time', ascending = True)

In [ ]:
biomes_hist

In [ ]:
biomes_hist.to_netcdf('/work/m1c/CMIP6_biome_PSS_data/biom_CMCC_PSS_hist_biovolume.nc')

In [ ]:
np.nanmean(biomes_hist.phymisc_per.values)

In [ ]:
biomes_hist.mean(dim=('time')).zmeso_per.plot(vmin=0, vmax=100)

## Code for revisions: subset biovolume for the size classes between the minimun PSSdb size (116 micrometers (823500.0 cubic microns)) and the shared maximum size represented across all models (2000 microns (4188790204.786391 cubic micrometers))

In [2]:
#load existing .nc file with the variables (No access to original carbon variables, so subsetted biovolume will be done by the NB field):
ds_hist = xr.open_mfdataset('/Users/mc4214/CMIP6_biome_PSS_data/biom_CMCC_PSS_hist_biovolume_old.nc')
ds_hist = ds_hist.rename({'total_biovolume_hist': 'total_biovolume_hist_not_standard'})
ds_hist['total_biovolume_hist_not_standard'].attrs = {"units": 'um^3 m^-2 ', 'Description': 'total biovolume from the size range of 116-20000 micrometers'}
ds_hist

<xarray.Dataset>
Dimensions:                            (lat: 180, lon: 360, time: 360,
                                        biovol_um3: 50)
Coordinates:
  * time                               (time) datetime64[ns] 1985-01-16 ... 2...
  * lat                                (lat) float64 -89.5 -88.5 ... 88.5 89.5
  * lon                                (lon) float64 -179.5 -178.5 ... 179.5
  * biovol_um3                         (biovol_um3) float64 5.939 ... 1.697e+13
Data variables: (12/19)
    biomes                             (lat, lon, time) float64 dask.array<chunksize=(180, 360, 360), meta=np.ndarray>
    chl                                (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    NB                                 (biovol_um3, time, lat, lon) float64 dask.array<chunksize=(50, 360, 180, 360), meta=np.ndarray>
    total_biovolume_hist_not_standard  (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    betas_hist                         (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    intercept_hist                     (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    ...                                 ...
    phymisc_per                        (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    phydiat_per                        (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    zmicro_per                         (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    zmeso_per                          (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    phyc                               (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    zooc                               (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>

In [3]:
#to properly backcalculate the biovolume per size class, it is necessary to get the bin widths from the ds_hist.biovol_um3 midpoints:
midpoints = ds_hist.biovol_um3
# Compute geometric means between midpoints for edges
log_mid = np.log10(midpoints.values)
log_edges_inner = 0.5 * (log_mid[1:] + log_mid[:-1])  # interior edges
len(log_edges_inner)

# Estimate outer edges by extrapolating
log_first_edge = log_mid[0] - (log_edges_inner[0] - log_mid[0])
log_last_edge = log_mid[-1] + (log_mid[-1] - log_edges_inner[-1])

log_edges = np.concatenate(([log_first_edge], log_edges_inner, [log_last_edge]))
edges = 10 ** log_edges

# Compute bin widths
bin_widths = np.diff(edges)
len(bin_widths)

50

In [4]:
bin_widths

array([3.52648695e+00, 6.33194321e+00, 1.13693165e+01, 2.04147261e+01,
       3.66559835e+01, 6.58183502e+01, 1.18181619e+02, 2.12203659e+02,
       3.81027483e+02, 6.84162828e+02, 1.22846526e+03, 2.20580060e+03,
       3.96067724e+03, 7.11168799e+03, 1.27695611e+04, 2.29286900e+04,
       4.11701550e+04, 7.39240519e+04, 1.32736092e+05, 2.38337451e+05,
       4.27952487e+05, 7.68420280e+05, 1.37975533e+06, 2.47745255e+06,
       4.44844892e+06, 7.98751841e+06, 1.43421789e+07, 2.57524409e+07,
       4.62404085e+07, 8.30280667e+07, 1.49083023e+08, 2.67689572e+08,
       4.80656385e+08, 8.63054020e+08, 1.54967720e+09, 2.78255980e+09,
       4.99629152e+09, 8.97121022e+09, 1.61084702e+10, 2.89239472e+10,
       5.19350823e+10, 9.32532738e+10, 1.67443137e+11, 3.00656512e+11,
       5.39850960e+11, 9.69342247e+11, 1.74052556e+12, 3.12524211e+12,
       5.61160291e+12, 1.00760472e+13])

In [5]:
# Make bin_widths into a DataArray with 'biovol_um3' coordinate 
bin_widths_da = xr.DataArray(
    bin_widths,
    dims=['biovol_um3'],
    coords={'biovol_um3': ds_hist.biovol_um3}
)

# Multiply NB by bin_widths_da — automatic broadcasting 
ds_hist['biovol_per_size_class'] = ds_hist.NB * bin_widths_da
ds_hist['biovol_per_size_class'].attrs = {"units": 'um^3 m^-2 ', 'Description': 'total biovolume per size class'}

In [6]:
# now get total biovolume from the size range:
ds_hist['total_biovolume_hist'] = (
    ds_hist.biovol_per_size_class
    .where(
        (ds_hist.biovol_per_size_class.biovol_um3 > 826400.0) &
        (ds_hist.biovol_per_size_class.biovol_um3 < 4188790204.786391)
    )
    .sum(dim='biovol_um3')
)
ds_hist['total_biovolume_hist'].attrs = {"units": 'um^3 m^-2 ', 'Description':'total biovolume from the size range of 116-2000 micrometers'}

In [7]:
ds_hist = ds_hist.compute()

In [8]:
# Define compression/chunking settings for each variable
from dask.diagnostics import ProgressBar
encoding = {}

for var in ds_hist.data_vars:
    shape = ds_hist[var].shape
    dims = ds_hist[var].dims
    
    # Choose reasonable chunk sizes (smaller than or equal to dimension sizes)
    chunk_sizes = tuple(min(10, s) for s in shape)

    encoding[var] = {
        'zlib': True,
        'complevel': 4,
        'chunksizes': chunk_sizes,
        '_FillValue': np.nan 
    }
# Now write the file with compression and chunking
with ProgressBar():
    ds_hist.to_netcdf(
        '/Users/mc4214/CMIP6_biome_PSS_data/biom_CMCC_PSS_hist_biovolume_new.nc',
        engine='netcdf4',
        encoding=encoding
    )

In [9]:
ds_hist.close()

<h1><center> SAME PROCESS BUT FOR SSP5 8.5</center></h1>

## Pre step: combine the different files of the plankton into one netcdf file

In [ ]:


# combine ssp5 data in one file and add variables by depth, and remove old datasets
ds_ssp585= xr.open_mfdataset(files_ssp585, combine = 'by_coords')
ds_ssp585 = ds_ssp585.drop_vars('zmeso200')
for i in [ 'phydiat', 'phymisc', 'zmicro', 'zmeso']:
    ds_ssp585[i +'_0_200'] = ds_ssp585[i +'_100'] + ds_ssp585[i+'_200']
    ds_ssp585 = ds_ssp585.drop_vars([i+'_100', i+'_200'])
ds_ssp585=ds_ssp585.sel(time= slice(2070.0, 2100)) # some data earlier than 1984 had to be removed
ds_ssp585=ds_ssp585.sortby('lat', ascending=True)
ds_ssp585

In [ ]:
#necessary step to make sure that cells with Nans are the same across the PFTs
ds_ssp585_mask = ~(np.isnan(ds_ssp585.phydiat_0_200) | np.isnan(ds_ssp585.phymisc_0_200) | np.isnan(ds_ssp585.zmicro_0_200) | np.isnan(ds_ssp585.zmeso_0_200))
ds_ssp585 = ds_ssp585.where(ds_ssp585_mask)

In [ ]:
total_carbon_gC_ssp5={}
total_carbon_gC_ssp5['globalSum_gC']={}
    
for i in varlist:
    total_carbon_gC_ssp5['globalSum_gC'][i] = (ds_ssp585[i + '_0_200'].mean(dim = 'time') * area_grid.areacello).sum(dim=['lon','lat']).values # conversion to g C already happened
total_carbon_gC_ssp5

In [ ]:
# transform the dictionary to a dataframe
biomass_ssp585=pd.DataFrame.from_dict(total_carbon_gC_ssp5)
biomass_ssp585=biomass_ssp585.reset_index()
biomass_ssp585.columns.values[0]='name'
biomass_ssp585

In [ ]:
global_um3_list = []
for n, v in enumerate(biomass_ssp585.name.unique()):
    global_um3_list.append(g_carbon_to_biovol(biomass_ssp585.globalSum_gC[n], v))
biomass_ssp585['globalSum_um3'] = global_um3_list

In [ ]:
biomass_ssp585

In [ ]:
sizedf_ssp585=pd.DataFrame([10**sizes]).transpose()
sizedf_ssp585.columns=['sizes']
sizedf_ssp585['phyto']= None
sizedf_ssp585['zoo']= None
#sizedf.loc[sizedf.sizes < 10,'phyto']='smp'
sizedf_ssp585.loc[(sizedf_ssp585.sizes > 2) & (sizedf_ssp585.sizes < 20),'phyto']='phymisc'
sizedf_ssp585.loc[(sizedf_ssp585.sizes > 20) & (sizedf_ssp585.sizes < 200),'phyto']='phydiat' ##Playing with diatom size 
sizedf_ssp585.loc[(sizedf_ssp585.sizes > 20) & (sizedf_ssp585.sizes < 200),'zoo']='zmicro'
sizedf_ssp585.loc[(sizedf_ssp585.sizes > 200) & (sizedf_ssp585.sizes < 35000),'zoo']='zmeso'

sizedf_ssp585

In [ ]:
# determine the degree of overlap between size bins. In COBALT (and perhaps other models) the degree of overlap might need to be determined following 
# Jessica's method

# the goal is to have repeated entries for a size class if it occurs across plantkon types:
sdfm_ssp585 = pd.melt(sizedf_ssp585, id_vars='sizes',var_name='type',value_name='name')
sdfm_ssp585 = sdfm_ssp585.dropna().reset_index(drop=True)
#pd.set_option('display.max_rows',80)
sdfm_ssp585

In [ ]:
# assign biovolume and amount of acrbon to each size bin
import math

sdfm_ssp585['biovolume_um3']=(4/3)*math.pi*(sdfm_ssp585.sizes/2)**3

sdfm_ssp585['mg_carbon']=0

# phymisc, which are non diatoms, treated as protists, Menden-Deuer and Lessard 2000
# < 3000 um3 biovolume
tmp=sdfm_ssp585.loc[(sdfm_ssp585.name=='phymisc') & (sdfm_ssp585.biovolume_um3 < 3000),'biovolume_um3']
sdfm_ssp585.loc[(sdfm_ssp585.name=='phymisc') & (sdfm_ssp585.biovolume_um3 < 3000),'mg_carbon']=10**(-0.583 + 0.860 * np.log10(tmp)) * 1e-9
# > 3000 um3 biovolume OJO under current size classes, all phydiat will fit here. This might be a big source of biomass overestimation
tmp=sdfm_ssp585.loc[(sdfm_ssp585.name=='phymisc') & (sdfm_ssp585.biovolume_um3 >= 3000),'biovolume_um3']
sdfm_ssp585.loc[(sdfm_ssp585.name=='phymisc') & (sdfm_ssp585.biovolume_um3 >= 3000),'mg_carbon']=10**(-0.665 + 0.939 * np.log10(tmp)) * 1e-9


# zmicro, treated as protists, Menden-Deuer and Lessard 2000
# < 3000 um3 biovolume
tmp=sdfm_ssp585.loc[(sdfm_ssp585.name=='zmicro') & (sdfm_ssp585.biovolume_um3 < 3000),'biovolume_um3']
sdfm_ssp585.loc[(sdfm_ssp585.name=='zmicro') & (sdfm_ssp585.biovolume_um3 < 3000),'mg_carbon']=10**(-0.583 + 0.860 * np.log10(tmp)) * 1e-9
# > 3000 um3 biovolume OJO under current size classes, all phydiat will fit here. This might be a big source of biomass overestimation
tmp=sdfm_ssp585.loc[(sdfm.name=='zmicro') & (sdfm_ssp585.biovolume_um3 >= 3000),'biovolume_um3']
sdfm_ssp585.loc[(sdfm_ssp585.name=='zmicro') & (sdfm_ssp585.biovolume_um3 >= 3000),'mg_carbon']=10**(-0.665 + 0.939 * np.log10(tmp)) * 1e-9


#sdfm.loc[sdfm.name=='phymisc','mg_carbon']=0.216 * sdfm.loc[sdfm.name=='phymisc','biovolume_um3']**0.939 * 1e-9
#sdfm.loc[sdfm.name=='zmicro','mg_carbon']=0.216 * sdfm.loc[sdfm.name=='zmicro','biovolume_um3']**0.939 * 1e-9

# diatoms, Menden-Deuer and Lessard 2000
# < 3000 um3 biovolume
tmp_ssp585=sdfm_ssp585.loc[(sdfm_ssp585.name=='phydiat') & (sdfm_ssp585.biovolume_um3 <= 3000),'biovolume_um3']
sdfm_ssp585.loc[(sdfm_ssp585.name=='phydiat') & (sdfm_ssp585.biovolume_um3 <= 3000),'mg_carbon']=10**(-0.541 + 0.811 * np.log10(tmp_ssp585)) * 1e-9
# > 3000 um3 biovolume OJO under current size classes, all phydiat will fit here. This might be a big source of biomass overestimation
tmp_ssp585=sdfm_ssp585.loc[(sdfm_ssp585.name=='phydiat') & (sdfm_ssp585.biovolume_um3 > 3000),'biovolume_um3']
sdfm_ssp585.loc[(sdfm_ssp585.name=='phydiat') & (sdfm_ssp585.biovolume_um3 > 3000),'mg_carbon']=10**(-0.933 + 0.881 * np.log10(tmp_ssp585)) * 1e-9

# mesozooplankton, Pitt et al. 2013 will be deprecated, we will use Kiorboe (2013)/Maas et al. (2021) combo
#sdfm.loc[sdfm.name=='zmeso','mg_carbon']= 0.06281 * (sdfm.loc[sdfm.name=='zmeso','sizes']/1e3)**3
sdfm_ssp585.loc[sdfm_ssp585.name=='zmeso','mg_carbon'] = 0.055 * (sdfm_ssp585.loc[sdfm.name=='zmeso','biovolume_um3']/1e9) # Maas et al. takes milimeters cubed (notice conversion) and returns dry mass in mg
sdfm_ssp585.loc[sdfm_ssp585.name=='zmeso','mg_carbon'] = 10**((np.log10(sdfm_ssp585.loc[sdfm_ssp585.name=='zmeso','mg_carbon'])-(-0.67))/0.96) # Kiorboe et al. takes dry mass in mg to wet mass in mg
sdfm_ssp585.loc[sdfm_ssp585.name=='zmeso','mg_carbon'] = (10**((0.95*np.log10(sdfm_ssp585.loc[sdfm_ssp585.name=='zmeso','mg_carbon']))-0.93))# Kiorboe et al. takes wet mass  and returns mass of carbon


## Step 3. pull together and merge the overlapping size bins. The total global carbon is split by the size bins.. yes?

In [ ]:

sdfm_ssp585['globalSum_um3_split']=0
for s in sdfm_ssp585.name.unique():
    n=len(sdfm_ssp585.loc[sdfm.name==s].index)
    print(s, n)
    sdfm_ssp585.loc[sdfm_ssp585.name==s,'globalSum_um3_split'] = np.tile(biomass_ssp585.loc[biomass_ssp585.name==s,'globalSum_um3']/n,n)
sdfm_ssp585

In [ ]:
sdfm_ssp585 = sdfm_ssp585.sort_values(by='biovolume_um3', ascending=True)


In [ ]:
sdfm_ssp585

In [ ]:
small_increment = (sdfm_ssp585['biovolume_um3'][1]-sdfm_ssp585['biovolume_um3'][0])/2 # small increment is used to define the maximum and minimum of the size rang

In [ ]:
# create log-spaced bins for mg_carbon
bins = np.logspace(np.log10(sdfm_ssp585['biovolume_um3'].min()-small_increment), np.log10(sdfm_ssp585['biovolume_um3'].max()+small_increment), 51)

# use pandas.cut to bin the data into log-spaced bins
sdfm_ssp585['biovolume_um3_bin'] = pd.cut(sdfm_ssp585['biovolume_um3'], bins=bins, include_lowest=False)
sdfm_ssp585['bin_centers'] = sdfm_ssp585['biovolume_um3_bin'].apply(lambda x:x.mid).astype(float) # this gets the mid point 
sdfm_ssp585['bin_range'] = sdfm_ssp585['biovolume_um3_bin'].apply(lambda x:x.length).astype(float)

len(sdfm_ssp585)

In [ ]:
df_grouped_ssp585 = sdfm_ssp585.groupby(['biovolume_um3_bin','bin_centers','bin_range', 'name']).agg(biovolume_um3=('biovolume_um3','mean'),
                                                                      sizes=('sizes','mean'))
df_grouped_ssp585 = df_grouped_ssp585.dropna().reset_index()
len(df_grouped_ssp585)

In [ ]:
df_grouped_ssp585

In [ ]:
bin_info_ssp585 = pd.DataFrame({
    'biovolume_um3_bin': np.sort(sdfm_ssp585['biovolume_um3_bin'].unique()),
    'bin_centers': np.sort(sdfm_ssp585['bin_centers'].unique())})

## Calculate  total biomass and normalized biomass per size class

In [ ]:
lat = ds_ssp585.lat
lon = ds_ssp585.lon
time = ds_ssp585.time
biovol_um3 = bin_info_ssp585.bin_centers
data_ssp585 = np.zeros((len(biovol_um3), len(time), len(lat), len(lon)))
biovolume_all_ssp585 = xr.DataArray(data_ssp585, coords={'biovol_um3':biovol_um3, 'time':time, 'lat':lat, 'lon':lon},
            dims = ['biovol_um3', 'time','lat', 'lon'])

lat_NB = ds_ssp585.lat
lon_NB = ds_ssp585.lon
time_NB = ds_ssp585.time
biovol_um3_NB = bin_info_ssp585.bin_centers
data_ssp585_NB = np.zeros((len(biovol_um3_NB), len(time_NB), len(lat_NB), len(lon_NB)))
biovolume_all_ssp585_NB = xr.DataArray(data_ssp585_NB, coords={'biovol_um3':biovol_um3_NB, 'time':time_NB, 'lat':lat_NB, 'lon':lon_NB},
            dims = ['biovol_um3', 'time','lat', 'lon'])

#time = ds.time
#data = np.zeros((len(time), len(mmolC), len(z_t_150m), len(nlat), len(nlon)))
#biomass_all = xr.DataArray(data, coords={'time':time, 'mass_mmolC':mmolC, 'z_t_150m':z_t_150m, 'nlat':nlat, 'nlon':nlon},
#            dims = ['time', 'mass_mmolC', 'z_t_150m', 'nlat', 'nlon'])


biovolume_all_ssp585.shape

In [ ]:
biovolume_all_ssp585_NB.shape

In [ ]:
biovolume_all_ssp585 = size_spectra(biovolume_all_ssp585,ds_ssp585, NB=False)

In [ ]:
biovolume_all_ssp585.mean(dim=('lat', 'lon', 'time')).plot()
plt.xscale('log')
plt.yscale('log')

## Get the total biovolume for only the size range included in PSSdb UVP+Zooscan

In [ ]:
biovol_mask_ssp585 = ~np.isnan(biovolume_all_ssp585.mean(dim=('biovol_um3')))
biovolume_all_subset_ssp585= biovolume_all_ssp585.where((biovolume_all_ssp585['biovol_um3']>826400.0) & (biovolume_all_ssp585['biovol_um3']<49100000000000.0)).sum(dim=['biovol_um3'])
biovolume_all_ssp585= biovolume_all_ssp585.sum(dim=['biovol_um3'])
biovolume_all_ssp585 = biovolume_all_ssp585.where(biovol_mask_ssp585)
biovolume_all_subset_ssp585 = biovolume_all_subset_ssp585.where(biovol_mask_ssp585)
biovolume_all_ssp585.mean(dim=('time')).plot()

                               

In [ ]:
biovolume_all_ssp585_NB = size_spectra(biovolume_all_ssp585_NB,ds_ssp585, NB=True)

In [ ]:
biovolume_all_ssp585_NB.mean(dim=('lat', 'lon', 'time')).plot()
plt.xscale('log')
plt.yscale('log')

In [ ]:
# removing data from mediterranean and black sea
#biovolume_all_ssp585.loc[dict(lat=biovolume_all_ssp585.coords['lat'][(biovolume_all_ssp585.coords['lat'] >= 30.5) & (biovolume_all_ssp585.coords['lat'] <= 47.5)],
                                        #lon=biovolume_all_ssp585.coords['lon'][(biovolume_all_ssp585.coords['lon'] >= -5.5) & (biovolume_all_ssp585.coords['lon'] <= 55.5)])]=float('nan')#

In [ ]:
betas_ssp5, intercept_ssp5, R2_ssp5, RMSE_ssp5 = calculate_size_spectra_slopes(biovolume_all_ssp585_NB)
betas_ssp5.shape

In [ ]:
np.nanmean(betas_ssp5)

In [ ]:
biovolume_all_ssp585_NB.values=10**(biovolume_all_ssp585_NB.values)

## add total  grid biovolume to the xarray and untransform NB values. The data needs to be coverted to biovolume

In [ ]:
#phymisc=g_carbon_to_biovol(((ds_ssp585.phymisc_0_200.values)*carbon_molar_mass), 'phymisc')
#phydiat=g_carbon_to_biovol(((ds_ssp585.phydiat_0_200.values)*carbon_molar_mass), 'phydiat')
#zmicro=g_carbon_to_biovol(((ds_ssp585.zmicro_0_200.values)*carbon_molar_mass), 'zmicro')
#zmeso=g_carbon_to_biovol(((ds_ssp585.zmeso_0_200.values)*carbon_molar_mass), 'zmeso')



## Get the slopes for each biome

In [ ]:
# load the biome mask
biome_mask = '/work/jyl/proj/CMIP6_models/ESM_Biomes/CMCC_ssp585_biomes_x1.nc'
biomes_ssp5 = xr.open_dataset(biome_mask)
biomes_mask = ~np.isnan(biomes_ssp5.biomes)
biomes_ssp5['biomes'] = biomes_ssp5['biomes'].where((biomes_ssp5['lat'] > -44.5) & (biomes_ssp5['lat'] < 44.5), 2)
biomes_ssp5 = biomes_ssp5.where(biomes_mask)

biomes_ssp5.biomes.plot()

In [ ]:
biomes_ssp5

In [ ]:

# add slopes to the biomes dataset
biomes_ssp5['chl'] = (('time', 'lat', 'lon'), chl_ssp5.schl.values)
biomes_ssp5['chl'].attrs = {"units": 'kg m-3', 'Description': 'Surface Mass Concentration of Total Phytoplankton expressed as Chlorophyll in Sea Water'}

biomes_ssp5['NB'] = biovolume_all_ssp585_NB
biomes_ssp5['NB'].attrs = {"units": 'um^3 m^-2 m^-3', 'Description': 'normalized biovolume for each size class'}

#biomes_ssp5['total_biovolume_ssp5_full'] = biovolume_all
biomes_ssp5['total_biovolume_ssp5'] = biovolume_all_subset_ssp585#(('time', 'lat', 'lon'),biovolume_all.values)
biomes_ssp5['total_biovolume_ssp5'].attrs = {"units": 'um^3 ', 'Description': 'total biovolume from the size range of 116-2000 micrometers'}

biomes_ssp5['betas_ssp5'] = (('time', 'lat', 'lon'), betas_ssp5)
biomes_ssp5['betas_ssp5'].attrs = {"units": 'm^-2 m^-3', 'Description': 'Normalized biovolume size spectra slope'}

biomes_ssp5['intercept_ssp5'] = (('time', 'lat', 'lon'), intercept_ssp5)
biomes_ssp5['intercept_ssp5'].attrs = {"units": 'um^3 m^-2 m^-3', 'Description': 'Normalized biovolume size spectra intercept'}

biomes_ssp5['R2_ssp5'] = (('time', 'lat', 'lon'), R2_ssp5)
biomes_ssp5['R2_ssp5'].attrs = { 'Description': 'Normalized biovolume coefficient of determination'}

biomes_ssp5['RMSE_ssp5'] = (('time', 'lat', 'lon'), RMSE_ssp5)
biomes_ssp5['RMSE_ssp5'].attrs = {"units": 'um^3 m^-2 m^-3', 'Description': 'Normalized biovolume size spectra Root mean square error'}

#original_biomass_variables
biomes_ssp5['phymisc'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_ssp585.phymisc_0_200.values*carbon_molar_mass, 'phymisc'))
biomes_ssp5['phymisc'].attrs = {"units": 'um^3', 'Description': 'miscellaneous phytoplankton biovolume'}

biomes_ssp5['phydiat'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_ssp585.phydiat_0_200.values*carbon_molar_mass, 'phydiat'))
biomes_ssp5['phydiat'].attrs = {"units": 'um^3', 'Description': 'diatom biovolume'}

biomes_ssp5['zmicro'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_ssp585.zmicro_0_200.values*carbon_molar_mass, 'zmicro'))
biomes_ssp5['zmicro'].attrs = {"units": 'um^3', 'Description': 'microzooplankton biovolume'}

biomes_ssp5['zmeso'] = (('time', 'lat', 'lon'),g_carbon_to_biovol(ds_ssp585.zmeso_0_200.values*carbon_molar_mass, 'zmeso'))
biomes_ssp5['zmeso'].attrs = {"units": 'um^3', 'Description': 'mesozooplankton biovolume'}

biomes_ssp5['total_biovolume_full']=(('time', 'lat', 'lon'), biomes_ssp5.phymisc.values+biomes_ssp5.phydiat.values+biomes_ssp5.zmicro.values+biomes_ssp5.zmeso.values)
biomes_ssp5['total_biovolume_full'].attrs = {"units": 'um^3', 'Description': 'all plankton biovolume'}
                                          
#percentage total
biomes_ssp5['phymisc_per'] = (('time', 'lat', 'lon'), (biomes_ssp5.phymisc.values/biomes_ssp5.total_biovolume_full.values)*100)
biomes_ssp5['phymisc_per'].attrs = {'Description': 'miscellaneous phytoplankton percentage of total biovolume'}                                        
                                          
biomes_ssp5['phydiat_per'] = (('time', 'lat', 'lon'),(biomes_ssp5.phydiat.values/biomes_ssp5.total_biovolume_full.values)*100)
biomes_ssp5['phydiat_per'].attrs = {'Description': 'diatom percentage of total biovolume'}                                           
                                          
biomes_ssp5['zmicro_per'] = (('time', 'lat', 'lon'),(biomes_ssp5.zmicro.values/biomes_ssp5.total_biovolume_full.values)*100)
biomes_ssp5['zmicro_per'].attrs = {'Description': 'microzooplankton percentage of total biovolume'} 

biomes_ssp5['zmeso_per'] = (('time', 'lat', 'lon'),(biomes_ssp5.zmeso.values/biomes_ssp5.total_biovolume_full.values)*100)
biomes_ssp5['zmeso_per'].attrs = {'Description': 'mesozooplankton percentage of total biovolume'} 

#total biovolume
biomes_ssp5['phyc'] = (('time', 'lat', 'lon'), biomes_ssp5.phymisc.values+biomes_ssp5.phydiat.values)
biomes_ssp5['phyc'].attrs = {"units": 'um^3', 'Description': 'phytoplankton biovolume'}

biomes_ssp5['zooc'] = (('time', 'lat', 'lon'), biomes_ssp5.zmicro.values+biomes_ssp5.zmeso.values)
biomes_ssp5['zooc'].attrs = {"units": 'um^3', 'Description': 'zooplankton biovolume'}

#biomes_ssp5['total_biovolume_ssp5']= (('time', 'lat', 'lon'), phypico+phydiat+zooc)
biomes_ssp5

In [ ]:
biomes_ssp5.mean(dim=('time', 'biovol_um3')).NB.plot()

## Change time format from 365_day to day-month-year

In [ ]:
biomes_ssp5=biomes_ssp5.assign_coords(year = np.trunc(biomes_ssp5.time).astype(int))#.astype(int)
biomes_ssp5=biomes_ssp5.assign_coords(day = ((biomes_ssp5.time - biomes_ssp5.year)*365).astype(int))
biomes_ssp5 = biomes_ssp5.where(biomes_ssp5.day !=0, drop=True)

biomes_ssp5=biomes_ssp5.assign_coords(year = biomes_ssp5.year.astype(str))
biomes_ssp5=biomes_ssp5.assign_coords(day = np.char.zfill(biomes_ssp5.day.astype(str),3))



In [ ]:
biomes_ssp5=biomes_ssp5.assign_coords(time = np.char.add(np.char.add(biomes_ssp5.year, '-'),biomes_ssp5.day))
#ds_hist=ds_hist.assign_coords(time = ds_hist.str.cat(ds_hist.year,ds_hist.day, sep = '-'))

In [ ]:
biomes_ssp5= biomes_ssp5.drop_dims('day')
biomes_ssp5 = biomes_ssp5.drop('year', dim=None)

In [ ]:
#sorted(ds_hist.time.values)

In [ ]:
from datetime import datetime as dt
biomes_ssp5=biomes_ssp5.assign_coords(time = [dt.strptime(x, '%Y-%j') for x in biomes_ssp5.time.values])

#ds_hist['time'].dt.strftime('%y%j')

In [ ]:
biomes_ssp5 = biomes_ssp5.sortby('time', ascending = True)

In [ ]:
biomes_ssp5

In [ ]:
biomes_ssp5.to_netcdf('/work/m1c/CMIP6_biome_PSS_data/biom_CMCC_PSS_ssp5_biovolume.nc')

## Code for revisions: subset biovolume for the size classes between the minimun PSSdb size (116 micrometers (823500.0 cubic microns)) and the shared maximum size represented across all models (2000 microns (4188790204.786391 cubic micrometers))

In [3]:
#load existing .nc file with the variables (No access to original carbon variables, so subsetted biovolume will be done by the NB field):
ds_ssp5 = xr.open_mfdataset('/Users/mc4214/CMIP6_biome_PSS_data/biom_CMCC_PSS_ssp5_biovolume_old.nc')
ds_ssp5 = ds_ssp5.rename({'total_biovolume_ssp5': 'total_biovolume_ssp5_not_standard'})
ds_ssp5['total_biovolume_ssp5_not_standard'].attrs = {"units": 'um^3 m^-2 ', 'Description': 'total biovolume from the size range of 116-20000 micrometers'}
ds_ssp5

<xarray.Dataset>
Dimensions:                            (lat: 180, lon: 360, time: 360,
                                        biovol_um3: 50)
Coordinates:
  * time                               (time) datetime64[ns] 2070-01-16 ... 2...
  * lat                                (lat) float64 -89.5 -88.5 ... 88.5 89.5
  * lon                                (lon) float64 -179.5 -178.5 ... 179.5
  * biovol_um3                         (biovol_um3) float64 5.939 ... 1.697e+13
Data variables: (12/19)
    biomes                             (lat, lon, time) float64 dask.array<chunksize=(180, 360, 360), meta=np.ndarray>
    chl                                (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    NB                                 (biovol_um3, time, lat, lon) float64 dask.array<chunksize=(50, 360, 180, 360), meta=np.ndarray>
    total_biovolume_ssp5_not_standard  (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    betas_ssp5                         (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    intercept_ssp5                     (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    ...                                 ...
    phymisc_per                        (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    phydiat_per                        (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    zmicro_per                         (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    zmeso_per                          (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    phyc                               (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>
    zooc                               (time, lat, lon) float64 dask.array<chunksize=(360, 180, 360), meta=np.ndarray>

In [4]:
#to properly backcalculate the biovolume per size class, it is necessary to get the bin widths from the ds_ssp5.biovol_um3 midpoints:
midpoints = ds_ssp5.biovol_um3
# Compute geometric means between midpoints for edges
log_mid = np.log10(midpoints.values)
log_edges_inner = 0.5 * (log_mid[1:] + log_mid[:-1])  # interior edges
len(log_edges_inner)

# Estimate outer edges by extrapolating
log_first_edge = log_mid[0] - (log_edges_inner[0] - log_mid[0])
log_last_edge = log_mid[-1] + (log_mid[-1] - log_edges_inner[-1])

log_edges = np.concatenate(([log_first_edge], log_edges_inner, [log_last_edge]))
edges = 10 ** log_edges

# Compute bin widths
bin_widths = np.diff(edges)
len(bin_widths)

50

In [5]:
bin_widths

array([3.52648695e+00, 6.33194321e+00, 1.13693165e+01, 2.04147261e+01,
       3.66559835e+01, 6.58183502e+01, 1.18181619e+02, 2.12203659e+02,
       3.81027483e+02, 6.84162828e+02, 1.22846526e+03, 2.20580060e+03,
       3.96067724e+03, 7.11168799e+03, 1.27695611e+04, 2.29286900e+04,
       4.11701550e+04, 7.39240519e+04, 1.32736092e+05, 2.38337451e+05,
       4.27952487e+05, 7.68420280e+05, 1.37975533e+06, 2.47745255e+06,
       4.44844892e+06, 7.98751841e+06, 1.43421789e+07, 2.57524409e+07,
       4.62404085e+07, 8.30280667e+07, 1.49083023e+08, 2.67689572e+08,
       4.80656385e+08, 8.63054020e+08, 1.54967720e+09, 2.78255980e+09,
       4.99629152e+09, 8.97121022e+09, 1.61084702e+10, 2.89239472e+10,
       5.19350823e+10, 9.32532738e+10, 1.67443137e+11, 3.00656512e+11,
       5.39850960e+11, 9.69342247e+11, 1.74052556e+12, 3.12524211e+12,
       5.61160291e+12, 1.00760472e+13])

In [6]:
# Make bin_widths into a DataArray with 'biovol_um3' coordinate 
bin_widths_da = xr.DataArray(
    bin_widths,
    dims=['biovol_um3'],
    coords={'biovol_um3': ds_ssp5.biovol_um3}
)

# Multiply NB by bin_widths_da — automatic broadcasting 
ds_ssp5['biovol_per_size_class'] = ds_ssp5.NB * bin_widths_da
ds_ssp5['biovol_per_size_class'].attrs = {"units": 'um^3 m^-2 ', 'Description': 'total biovolume per size class'}

In [7]:
# now get total biovolume from the size range:
ds_ssp5['total_biovolume_ssp5'] = (
    ds_ssp5.biovol_per_size_class
    .where(
        (ds_ssp5.biovol_per_size_class.biovol_um3 > 826400.0) &
        (ds_ssp5.biovol_per_size_class.biovol_um3 < 4188790204.786391)
    )
    .sum(dim='biovol_um3')
)
ds_ssp5['total_biovolume_ssp5'].attrs = {"units": 'um^3 m^-2 ', 'Description':'total biovolume from the size range of 116-2000 micrometers'}

In [8]:
ds_ssp5 = ds_ssp5.compute()

In [9]:
# Define compression/chunking settings for each variable
encoding = {}

for var in ds_ssp5.data_vars:
    shape = ds_ssp5[var].shape
    dims = ds_ssp5[var].dims
    
    # Choose reasonable chunk sizes (smaller than or equal to dimension sizes)
    chunk_sizes = tuple(min(50, s) for s in shape)

    encoding[var] = {
        'zlib': True,
        'complevel': 4,
        'chunksizes': chunk_sizes,
        '_FillValue': np.nan 
    }
# Now write the file with compression and chunking
ds_ssp5.to_netcdf(
    '/Users/mc4214/CMIP6_biome_PSS_data/biom_CMCC_PSS_ssp5_biovolume_new.nc',
    engine='netcdf4',
    encoding=encoding
)

In [10]:
ds_ssp5.close()

## include code to assess the size spectra

In [ ]:
from wcmatch.pathlib import Path # Handling of path object

In [ ]:
test_df = biomes_hist.mean(dim=['time']).to_dataframe()
test_df=test_df.dropna().reset_index()
test_df=test_df.drop_duplicates().reset_index(drop=True)
test_df['biomes'] = test_df['biomes'].astype(str)
test_df = test_df.replace({'biomes':{'1.0':'LC','2.0':'HCSS','3.0':'HCPS'}})

In [ ]:
import math as m
test_df['source']='CMCC'
test_df['ECD'] = ((test_df['biovol_um3']*6)/m.pi)**(1./3.)
test_df=test_df.astype(dict(zip(['biovol_um3', 'ECD'],[str]*2))).groupby(['source', 'biovol_um3', 'ECD']).apply(lambda x: pd.Series({'NB':np.nanmean(x.NB)})).reset_index()
test_df['biovol_um3']=test_df['biovol_um3'].astype(float)
test_df['ECD']=test_df['ECD'].astype(float)
test_df

In [ ]:
bins_df = pd.read_csv('/work/m1c/CMIP6_biome_PSS_data/ecopart_size_bins.csv', sep = ',')
test_df.loc[:, 'sizeClasses']= pd.cut(x=test_df['ECD'], bins=bins_df['ESD_um'], include_lowest=True)# size classes defined by biovolume
test_df['ECD'] = test_df.sizeClasses.apply(lambda x: x.mid)
test_df=test_df.drop(columns=['sizeClasses']).reset_index()
test_df['NB']=(test_df['NB']/200)*0.001# conversion to liters

In [ ]:
from plotnine import *
colors = dict(CESM= 'red',CMCC= 'cyan', CNRM= 'gray',GFDL= 'lawngreen',IPSL= 'gold',UKESM= 'purple', PSSdb = 'blue')
breaks = [20, 200, 2000, 20000]#breaks = [1, 2, 20, 200]#
labels = [20, 200, 2000, 20000]#labels = [1, 2, 20, 200]#
theme_paper=theme(axis_ticks_direction="inout",
              panel_grid=element_blank(),
              axis_line = element_line(colour = "black"),
              panel_background=element_rect(fill='white'),
              panel_border=element_blank(),
              legend_title=element_text(family="serif", size=15),
              legend_position='none',#legend_position='top',
              legend_text=element_text(family="serif", size=15),
              axis_title=element_text(family="serif", size=15),
              axis_text_x=element_text(family="serif", size=15),
              axis_text_y=element_text(family="serif", size=15, rotation=90),
              plot_background=element_rect(fill='white'), strip_background=element_rect(fill='white'))

In [ ]:
test_df['ECD']=test_df['ECD'].astype(float)

In [ ]:
plot = (ggplot(data=test_df)+
        geom_line(test_df,aes(x='ECD', y='NB', color='source', group='source'),  size = 1) +
        geom_point(aes(x='ECD', y='NB',color='source', group='source'),size = 3,  shape = 'o')+
        #stat_summary(data=df_NB[df_NB.ECD.transform(lambda x: x.astype(str).isin(pd.Series(x.value_counts(normalize=True)[x.value_counts(normalize=True)>=np.quantile(x.value_counts(normalize=True),0.5)].index).astype(str)))],mapping=aes(x='ECD', y='NB'),geom='line', fun_y=np.nanmedian, size = 0.5, alpha=0.5)+
        #stat_summary(mapping=aes(x='ECD', y='NB'),geom='line', fun_y=np.nanmedian, size = 0.8)+
        labs(y=r'Normalized Biovolume ($\mu$m$^{3}$ m$^{-2}$ $\mu$m$^{-3}$)', x=r'Equivalent circular diameter ($\mu$m)')+
        scale_color_manual(values = colors)+
        scale_y_log10(breaks=[10**np.arange(-5,10,step=2, dtype=np.float)][0],labels=['10$^{%s}$'% int(n) for n in np.arange(-5,10,step=2)] , limits=(1e-5, 1e11))+
        #scale_y_log10(breaks=[10**np.arange(-5,-3,step=0.5, dtype=np.float)][0],labels=['10$^{%s}$'% int(n) for n in np.arange(-5,-3,step=0.5)] , limits=(1e-5, 1e-3))+
        scale_x_log10(breaks=breaks, labels=labels,limits=(0.5, 35000))+
        #scale_x_log10(breaks=[size  for size in np.sort( np.concatenate(np.arange(1, 10).reshape((9, 1)) * np.power(10, np.arange(1, 5, 1))))],labels= [size if (size / np.power(10, np.ceil(np.log10(size)))) == 1 else '' for size in np.sort( np.concatenate(np.arange(1, 10).reshape((9, 1)) * np.power(10, np.arange(1, 5, 1))))], limits=(1, 15000))+
        theme_paper).draw(show=False)
plot.set_size_inches(6,6)
plot.savefig(fname='{}/work/CMIP6_size_spectra_scripts/plots/NBSS_CMCC_mean.pdf'.format(str(Path.home())), dpi=300)